In [ ]:
from dataclasses import dataclass, asdict
from typing import List, Optional
import json
import requests

In [ ]:
#!pip install requests beautifulsoup4

In [ ]:
@dataclass
class GrammarPoint:
    grammar: str
    meaning: str
    jlpt_level: str
    example_jp: Optional[str] = None
    example_en: Optional[str] = None

In [ ]:
def fetch_html(url: str) -> str:
    """Fetch HTML content from URL"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    print(f"Fetching {url}...")
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    response.encoding = 'utf-8'
    
    print(f"Successfully fetched {len(response.text)} characters")
    return response.text

In [ ]:
test_html = fetch_html("https://jlptgrammarlist.neocities.org/")

In [ ]:
def parse_grammar_from_html(html: str) -> List[GrammarPoint]:
    """Parse grammar points directly from the HTML DOM structure.
    
    The page has: <div class="grammar-list n5/n4/n3/n2/n1">
      containing <div class="item"> elements, each with:
        - <span class="term">   -> grammar point
        - loose text node       -> English meaning
        - <div class="japanese-sentence"> -> example sentence (JP)
        - <div class="english-meaning">   -> example sentence (EN)
    """
    from bs4 import BeautifulSoup
    
    soup = BeautifulSoup(html, 'html.parser')
    grammar_points = []
    
    level_map = {'n5': 'N5', 'n4': 'N4', 'n3': 'N3', 'n2': 'N2', 'n1': 'N1'}
    
    for level_class, level_name in level_map.items():
        level_div = soup.find('div', class_=level_class)
        if not level_div:
            continue
        
        for item in level_div.find_all('div', class_='item'):
            # Extract grammar term
            term_span = item.find('span', class_='term')
            if not term_span:
                continue
            grammar = term_span.get_text(strip=True)
            if not grammar:
                continue
            
            # Extract English meaning (loose text between <span class="common"> and <div>)
            # Get all direct text nodes that aren't inside child elements
            meaning_parts = []
            for child in item.children:
                if isinstance(child, str) and child.strip():
                    meaning_parts.append(child.strip())
            meaning = ' '.join(meaning_parts)
            
            if not meaning:
                continue
            
            # Extract example Japanese sentence
            jp_div = item.find('div', class_='japanese-sentence')
            example_jp = jp_div.get_text(strip=True) if jp_div else None
            
            # Extract example English meaning
            en_div = item.find('div', class_='english-meaning')
            example_en = en_div.get_text(strip=True) if en_div else None
            
            grammar_points.append(GrammarPoint(
                grammar=grammar,
                meaning=meaning,
                jlpt_level=level_name,
                example_jp=example_jp,
                example_en=example_en
            ))
    
    return grammar_points

In [ ]:
parsed_grammar = parse_grammar_from_html(test_html)
parsed_grammar

In [ ]:
# A Japanese-capable zero-shot classifier

# Labels dataset:
grammar_labels = [gp.grammar for gp in parsed_grammar]

# https://huggingface.co/akiFQC/bert-base-japanese-v3_nli-jsnli
classifier = pipeline("zero-shot-classification", model='akiFQC/bert-base-japanese-v3_nli-jsnli')
example_sents = [seg['text'] for seg in example_podcast]

# for sent in example_sents:
#     # Only classify labels whose grammar string appears in the sentence
#     candidates = [gp.grammar for gp in parsed_grammar if gp.grammar in sent]
#     if candidates:
#         res = classifier(sent, candidates, multi_label=True, batch_size=64)

sent = "Appleは先程、iPhoneの最新機種について発表しました。"
# candidate_labels = ["技術", "スポーツ", "政治"]
candidate_labels = grammar_labels
res = classifier(example_sents[0:10], candidate_labels, multi_label=True, batch_size=64, device=0)
res
# Zero-shot does not work very well for this.
# One-shot also does not work very well for this.
# 700 labels is too much. I think something I can do is I can break this up into different models.
# So split across all JLPT levels for all labels, which cuts the amount of labels by /5
# So the workflow would to be:
# 1. User downloads and parses their podcast
# 2. User then selects which grammar level to parse. EX: JLPT 1, 2, 3, 4, or 5  
#    2a. OR a combination of (1,2) (2,3) etc
#    2b. OR the user can input their own grammar patterns (this is a stretch feature for now)
# 3. The model runs a classification on the podcast sentences
# 4. How do I handle the data? Figure this out after a rough demonstative test

In [ ]:
# ===== BUILDING THE FEW-SHOT TEST DATASET ===== #

# ===== 10 Example Sentences per Grammar Point from JLPTSensei PDFs ===== #
from jlpt_grammar_parser import parse_jlpt_grammar_pdf, print_stats
import pandas as pd

# Define your PDF paths
pdfs = {
    'N5': '/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPT N5 Grammar Master Ebok by JLPTsensei.com.pdf',
    'N4': '/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPT N4 Grammar Master Ebook by JLPTsensei.com.pdf',
    'N3': '/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPT N3 Grammar Master Ebook by JLPTsensei.com.pdf',
    'N2': '/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPT N2 Grammar Master Ebook by JLPTsensei.com.pdf',
}

all_sentences = []
for level, path in pdfs.items():
    result = parse_jlpt_grammar_pdf(path, level=level)
    print_stats(result)
    all_sentences.extend(result['sentences'])

df = pd.DataFrame(all_sentences)
print(f"Total: {len(df)} sentences")

In [ ]:
# Store in CSV so I don't have to rerun the parser a lot
#df.to_csv("N5-N2_JLPTSensei.csv")
df = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N5-N2_JLPTSensei.csv")